# Решения: практика key + pointers

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import time
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


unsorted_df = pd.read_csv(_find('bank_transactions_unsorted.csv'))
by_id_df = pd.read_csv(_find('bank_transactions_sorted_by_txn_id.csv'))
by_amount_df = pd.read_csv(_find('bank_transactions_sorted_by_amount.csv'))
tiny_df = pd.read_csv(_find('bank_transactions_tiny.csv'))

unsorted_txns = list(unsorted_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_txns = list(by_id_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_list = [t[0] for t in id_txns]
amount_list = [t[1] for t in amount_txns]


In [ ]:
def merge_rows_by_amount(a_rows, b_rows):
    i = 0
    j = 0
    out = []
    while i < len(a_rows) and j < len(b_rows):
        if a_rows[i][1] <= b_rows[j][1]:
            out.append(a_rows[i]); i += 1
        else:
            out.append(b_rows[j]); j += 1
    out.extend(a_rows[i:])
    out.extend(b_rows[j:])
    return out


def min_gap_between_windows(a_amounts, b_amounts):
    i = 0
    j = 0
    best = abs(a_amounts[0] - b_amounts[0])
    while i < len(a_amounts) and j < len(b_amounts):
        cur = abs(a_amounts[i] - b_amounts[j])
        if cur < best:
            best = cur
        if a_amounts[i] < b_amounts[j]:
            i += 1
        else:
            j += 1
    return best


def binary_search_txn(sorted_ids, target_id):
    left, right = 0, len(sorted_ids) - 1
    while left <= right:
        mid = (left + right) // 2
        if sorted_ids[mid] == target_id:
            return mid
        if sorted_ids[mid] < target_id:
            left = mid + 1
        else:
            right = mid - 1
    return -1


left_rows = amount_txns[:40]
right_rows = amount_txns[40:80]
merged = merge_rows_by_amount(left_rows, right_rows)
w1 = amount_list[50:130]
w2 = amount_list[420:500]
gap = min_gap_between_windows(w1, w2)
targets = [id_list[10], id_list[90], id_list[190], id_list[390]]
positions = [binary_search_txn(id_list, x) for x in targets]
PRACTICE_NOTE = (
    'Один и тот же принцип двух указателей помогает и при слиянии, и при поиске близких сумм. '
    'Ключевое условие - данные должны быть отсортированы по рабочему признаку.'
)
tiny_rows = list(tiny_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
tiny_sorted = sorted(tiny_rows, key=lambda row: (row[2], row[1]))
tiny_amounts = sorted([x[1] for x in tiny_rows])
i = 0
j = len(tiny_amounts) - 1
pairs_cnt = 0
while i < j:
    if tiny_amounts[i] + tiny_amounts[j] >= 45000:
        pairs_cnt += j - i
        j -= 1
    else:
        i += 1
MINI_REPORT = (
    'На mini-логе отсортировали транзакции по дню и сумме для операционного просмотра. '
    'Двумя указателями оценили число пар крупных операций от 45000, '
    'что полезно как быстрый индикатор концентрации больших платежей.'
)
print(merged[:3])
print(gap)
print(positions)